# Phase 3 — Fold-0 CNN training (local / Lightning AI)

Trains the CNN encoder + cross-attention decoder on fold 0 of the Phase 1/2
tag-grouped CV. Run from inside a checked-out copy of the repo. Expects:

- This notebook lives in `notebooks/` of a cloned repo.
- Raw competition data sits at `<repo>/data/train/` (`*__horizontal_well.csv`,
  `*__typewell.csv`).
- A CUDA-capable GPU is available (Lightning AI Studio T4 / A10 / A100 etc.).

Outputs `fold_models/fold_0.pt` and `oof_fold_0.parquet` to
`<repo>/artefacts/nn/cnn/`. Download both for the eventual submission step.


In [ ]:
# Install runtime deps if the environment doesn't already have them.
# Lightning AI Studios usually ship with torch + pyarrow preinstalled; this is
# a no-op in that case.
%pip install -q torch pyarrow

In [ ]:
import os, sys
from pathlib import Path

# This notebook lives at <repo>/notebooks/. cd to <repo> so `python -m src.nn.cli`
# resolves the package and DATA_DIR / ARTEFACT_DIR default to repo-relative paths.
NB_DIR = Path.cwd()
REPO = NB_DIR if (NB_DIR / "src" / "nn" / "cli.py").exists() else NB_DIR.parent
assert (REPO / "src" / "nn" / "cli.py").exists(), f"Could not find repo root from {NB_DIR}"
os.chdir(REPO)
sys.path.insert(0, str(REPO))

print("Repo root:", REPO)
print("Has data/train:", (REPO / "data" / "train").exists())
print("First few wells:", sorted((REPO / "data" / "train").glob("*__horizontal_well.csv"))[:3])

In [ ]:
# DATA_DIR and ARTEFACT_DIR default to <REPO>/data and <REPO>/artefacts when not set.
os.environ["MODE"] = "fold"
os.environ["MODEL_KIND"] = "cnn"
os.environ["FOLD"] = "0"
os.environ["N_EPOCHS"] = "50"
os.environ["BATCH_SIZE"] = "16"
os.environ["DEVICE"] = "cuda"

import torch
print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"GPU memory: {free/2**30:.1f} GiB free / {total/2**30:.1f} GiB total")

In [ ]:
!python -m src.nn.cli

In [ ]:
# Inspect outputs.
out = REPO / "artefacts" / "nn" / "cnn"
for path in sorted(out.rglob("*")):
    if path.is_file():
        print(f"{path.relative_to(REPO)}  ({path.stat().st_size / 2**20:.2f} MiB)")
    else:
        print(f"{path.relative_to(REPO)}/")